In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        os.path.join(dirname, filename)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import random
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch import nn
from transformers import CLIPProcessor, CLIPModel
from transformers import CLIPImageProcessor, AutoTokenizer

from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- Configuration ---
# Set device to GPU if available, otherwise CPU
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Hyperparameters
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 1e-4

# Dataset path
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images/"

In [ ]:
# --- Data Preparation ---

# Get the base path for the dataset
DATASET_PATH = "/kaggle/input/ai-generated-images-vs-real-images/"

# --- CORRECTED: Point to the 'train' and 'test' directories ---
train_paths = []
val_paths = []

# Populate training paths
train_real_path = os.path.join(DATASET_PATH, 'train/real')
for img_name in os.listdir(train_real_path):
    train_paths.append((os.path.join(train_real_path, img_name), 0)) # Label 0 for Real

train_fake_path = os.path.join(DATASET_PATH, 'train/fake')
for img_name in os.listdir(train_fake_path):
    train_paths.append((os.path.join(train_fake_path, img_name), 1)) # Label 1 for Fake

# Populate validation paths from the 'test' directory
val_real_path = os.path.join(DATASET_PATH, 'test/real')
for img_name in os.listdir(val_real_path):
    val_paths.append((os.path.join(val_real_path, img_name), 0)) # Label 0 for Real

val_fake_path = os.path.join(DATASET_PATH, 'test/fake')
for img_name in os.listdir(val_fake_path):
    val_paths.append((os.path.join(val_fake_path, img_name), 1)) # Label 1 for Fake

# Shuffle the datasets for randomness
random.shuffle(train_paths)
random.shuffle(val_paths)

print(f"Training samples: {len(train_paths)}")
print(f"Validation samples: {len(val_paths)}")


# --- Custom PyTorch Dataset ---
class ImageDataset(Dataset):
    def __init__(self, image_paths, processor):
        self.image_paths = image_paths
        self.processor = processor

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path, label = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert("RGB")
            # Process the image using the CLIP processor
            processed_image = self.processor(images=image, return_tensors="pt")['pixel_values'].squeeze(0)
            return processed_image, torch.tensor(label, dtype=torch.float32)
        except Exception as e:
            # Handle potential corrupted images
            print(f"Skipping corrupted image: {img_path}, error: {e}")
            # Return the first image as a placeholder to avoid crashing the loader
            return self.__getitem__(0)

# Load the CLIP processor
clip_model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(clip_model_name)

# Create Dataset and DataLoader instances
train_dataset = ImageDataset(train_paths, processor)
val_dataset = ImageDataset(val_paths, processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# pip install transformers==4.30.2

In [ ]:
# # --- CORRECTED CODE to handle library updates ---
# # Load the CLIP processor's components separately
# from transformers import CLIPImageProcessor, CLIPTokenizer, CLIPProcessor

# clip_model_name = "openai/clip-vit-base-patch32"

# # Use the specific CLIPTokenizer class to avoid the error
# tokenizer = CLIPTokenizer.from_pretrained(clip_model_name)
# image_processor = CLIPImageProcessor.from_pretrained(clip_model_name)

# # Manually combine them into a CLIPProcessor instance
# processor = CLIPProcessor(image_processor=image_processor, tokenizer=tokenizer)

# Now you can use the processor as intended
print("Processor loaded successfully!")
print(processor)

In [ ]:
# --- Model Architecture ---
class CLIPImageClassifier(nn.Module):
    def __init__(self, clip_model_name="openai/clip-vit-base-patch32"):
        super(CLIPImageClassifier, self).__init__()
        # Load the pretrained CLIP model
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        # Freeze all the parameters in the CLIP model
        for param in self.clip.parameters():
            param.requires_grad = False

        # Define a custom classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.clip.config.vision_config.hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 1),
            nn.Sigmoid()  # Sigmoid for binary classification
        )

    def forward(self, pixel_values):
        # Get the image features from the CLIP vision model
        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        # The pooler_output gives a summary feature vector for the image
        image_features = vision_outputs.pooler_output
        
        # Pass the features through our custom classifier
        return self.classifier(image_features)

# Instantiate the model and move it to the GPU
model = CLIPImageClassifier().to(DEVICE)

In [ ]:
!nvidia-smi

In [ ]:
# --- Training ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

model.to(DEVICE)

# Optimizer and Loss Function
# We only pass the parameters of the trainable classifier head to the optimizer
optimizer = Adam(model.classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for images, labels in tqdm(dataloader, desc="Training"):
        images, labels = images.to(device), labels.to(device).unsqueeze(1)
        
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(images)
        
        # Calculate loss
        loss = criterion(outputs, labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating"):
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            
            # Calculate accuracy
            predicted = (outputs > 0.5).float()
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            
    accuracy = correct_predictions / total_samples
    return total_loss / len(dataloader), accuracy

best_val_accuracy = 0.0  # Initialize a tracker for the best accuracy
SAVE_PATH = "/kaggle/working/best_clip_finetuned_classifier.pth" # Path for the best model

# --- Main Loop ---
for epoch in range(EPOCHS):
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_loss, val_accuracy = validate(model, val_loader, criterion, DEVICE)
    
    print(f"Epoch {epoch+1} Summary:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Validation Loss: {val_loss:.4f}")
    print(f"  Validation Accuracy: {val_accuracy:.4f}")

    # --- Check if the current model is the best one and save it ---
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        # Save the state dictionary of the best model
        torch.save(model.state_dict(), SAVE_PATH)
        print(f"🎉 New best model saved! Accuracy: {val_accuracy:.4f} at Epoch {epoch+1}")
        print(f"   Model saved to {SAVE_PATH}")


# Save the trained model (optional)
# torch.save(model.state_dict(), 'clip_finetuned_classifier.pth')
print("\nTraining complete.")
print(f"The best model with accuracy {best_val_accuracy:.4f} is saved at {SAVE_PATH}")
